# Check: local-only release

Run with the **"ddi (5090)"** kernel after `bash setup/setup_local.sh`. Checks that

1. the `ddi` package is importable from the repo's venv, and torch, DGL and the GPU work,
2. no Colab code or text is left anywhere in the repo,
3. `get_paths` rejects unedited placeholders and expands `~`,
4. `download_raw_data` takes the database paths from `ddi.paths`, and the file names live only there,
5. (optional) your real `DATA_ROOT` has the expected inputs.

Every check cell ends with `OK`. The last cell runs `tests/test_local_only.py`.

In [ ]:
# ============================== EDIT THIS CELL ==============================
# Optional: your real roots. Leave the placeholders to skip check 5.
DATA_ROOT = "<DATA_ROOT: e.g. /home/you/ddi_data>"
WORK_ROOT = "<WORK_ROOT: e.g. /home/you/ddi_runs>"
# =============================================================================

## 1. Environment

In [ ]:
import os, sys, json, re, inspect, pathlib, tempfile, subprocess
import ddi
REPO_DIR = pathlib.Path(ddi.__file__).resolve().parents[2]
print("python  :", sys.executable)
print("ddi from:", pathlib.Path(ddi.__file__).parent)
print("repo    :", REPO_DIR)
if ".venv" not in sys.executable:
    print("WARNING: this kernel is not the repo's .venv; select the \"ddi (5090)\" kernel")

import torch, dgl
print("torch", torch.__version__, "| dgl", dgl.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    a = torch.randn(1024, 1024, device="cuda"); float((a @ a).sum())   # fails here on a wrong CUDA build
print("OK")

## 2. No Colab traces
Cell sources of every notebook, plus `.py`, `.sh`, `.md`, `.toml` files. This notebook and `tests/test_local_only.py` are skipped because they contain the search patterns.

In [ ]:
FORBIDDEN = re.compile(r"colab|/content|MyDrive|ON_COLAB|drive\.mount|google\.colab", re.I)
SKIP = {"tests/test_local_only.py", "notebooks/checks/local_only.ipynb",
        "notebooks/checks/db_tests.ipynb"}
hits, n_files = [], 0
for p in REPO_DIR.rglob("*"):
    rel = p.relative_to(REPO_DIR).as_posix()
    if (not p.is_file() or rel in SKIP or any(x.startswith(".") for x in p.parts)
            or "__pycache__" in rel or ".egg-info" in rel):
        continue
    if p.suffix == ".ipynb":
        text = "\n".join("".join(c["source"]) for c in json.loads(p.read_text())["cells"])
    elif p.suffix in {".py", ".sh", ".md", ".toml", ".txt", ".cfg", ".yaml", ".yml"}:
        text = p.read_text()
    else:
        continue
    n_files += 1
    for i, line in enumerate(text.splitlines(), 1):
        if FORBIDDEN.search(line):
            hits.append(f"{rel}:{i}: {line.strip()[:100]}")
print(f"scanned {n_files} files")
print("\n".join(hits) if hits else "no Colab traces")
assert not (REPO_DIR / "setup" / "colab_setup.sh").exists(), "setup/colab_setup.sh still exists"
assert not hits, f"{len(hits)} Colab trace(s) left"
print("OK")

## 3. `get_paths`

In [ ]:
from ddi.paths import get_paths
tmp = pathlib.Path(tempfile.mkdtemp())
try:
    get_paths("<DATA_ROOT: e.g. /home/you/ddi_data>", str(tmp))
    raise AssertionError("an unedited placeholder was accepted")
except ValueError as e:
    print("placeholder rejected:", e)
p = get_paths("~/ddi_data_example", str(tmp))
print("~ expanded to:", p.data_root)
assert not p.data_root.startswith("~")
print("drug_db :", p.drug_db)
print("event_db:", p.event_db)
print("OK")

## 4. Raw databases: names only in `ddi.paths`, `download_raw_data` takes the paths
No network: the two files are created empty in a temporary folder, so the function must return without downloading.

In [ ]:
from ddi.data import download_raw_data
params = list(inspect.signature(download_raw_data).parameters)
print("download_raw_data parameters:", params)
assert params == ["drug_db", "event_db", "gdrive_id"]

for name in ("Drug_META_DDIE.db", "event_META_DDIE_5foldLabel.db"):
    where = sorted(f.relative_to(REPO_DIR).as_posix() for f in (REPO_DIR / "src").rglob("*.py")
                   if name in f.read_text())
    print(f"{name:<32} found in {where}")
    assert where == ["src/ddi/paths.py"]

p = get_paths(str(tmp / "in"), str(tmp / "out"))
os.makedirs(p.raw_data_dir)
open(p.drug_db, "w").close(); open(p.event_db, "w").close()
assert download_raw_data(p.drug_db, p.event_db, "unused-id") == (p.drug_db, p.event_db)
print("OK")

## 5. Your real roots (optional)
Reports what exists. Nothing is created in `DATA_ROOT`.

In [ ]:
if DATA_ROOT.startswith("<") or WORK_ROOT.startswith("<"):
    print("placeholders not set: skipped")
else:
    paths = get_paths(DATA_ROOT, WORK_ROOT)
    paths.show()
    print()
    for label, f in [("thesis graph zip", paths.graph_zip), ("drug database", paths.drug_db),
                     ("event database", paths.event_db), ("kNN cache k=5", paths.thesis_knn_cache_k5)]:
        print(f"  {label:<18} {'present' if os.path.exists(f) else 'MISSING'}   {f}")
    if os.path.exists(paths.graph_bin):
        from ddi.graph import TRUNCATED_BIN_SIZE
        size = os.path.getsize(paths.graph_bin)
        print(f"  extracted .bin     {size:,} bytes" + ("   TRUNCATED: delete it" if size == TRUNCATED_BIN_SIZE else ""))
print("OK")

## 6. Unit tests

In [ ]:
r = subprocess.run([sys.executable, "-m", "pytest", "-q", str(REPO_DIR / "tests" / "test_local_only.py")],
                   capture_output=True, text=True, cwd=REPO_DIR)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "tests/test_local_only.py failed"
print("ALL CHECKS PASSED")